how does my sleep effect my activites the following day!!

parituclraly like my phisology - does stride length, cadence change

does heart rate, min/max, avg, hrv change?

is my training load at the same intesity workout changed?


In [6]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import sqlite3
import pandas as pd
import numpy as np

from scipy.stats import pearsonr

from src.utils.feature_engineering import add_sleep_features

In [3]:
con = sqlite3.connect(r"C:\Users\Dexta\Learning\Garmin-Analysis\data\garmin.db")

activities_df = pd.read_sql(
    "SELECT * FROM ActivitySummary",
    con
)

sleep_df = pd.read_sql(
    "SELECT * FROM Sleep",
    con
)



In [9]:
activities_df.columns

Index(['activity_id', 'startTimeGmt', 'calendarDate', 'start_time',
       'activity_type', 'durationSeconds', 'distanceMeters',
       'ratio_time_max_hr', 'ratio_time_avg_hr', 'hr_volatility',
       'hr_derivative', 'hr_smoothness', 'hr_oscillation', 'hr_drift',
       'meters_per_beat', 'plateau_seconds', 'hr_pace_coupling',
       'start_hr_offset', 'cadence_std', 'cadence_cv', 'cadence_drift',
       'cadence_derivative', 'cadence_smoothness', 'cadence_speed_coupling',
       'cadence_hr_coupling', 'steps_per_meter', 'running_start_cadence',
       'avg_speed', 'max_speed', 'speed_std', 'speed_cv', 'speed_drift',
       'speed_derivative', 'speed_smoothness', 'speed_oscillation',
       'speed_hr_coupling', 'speed_cadence_coupling', 'running_start_speed',
       'stride_std', 'stride_cv', 'stride_drift', 'stride_derivative',
       'stride_smoothness', 'stride_oscillation', 'stride_speed_coupling',
       'stride_hr_coupling', 'running_start_stride', 'avgHr', 'maxHr',
       'avg

In [7]:

# Align sleep + daily on calendarDate
sleep = add_sleep_features(sleep_df)

activities_df["calendarDate"] = pd.to_datetime(activities_df["calendarDate"])
sleep["calendarDate"] = pd.to_datetime(sleep["calendarDate"])

merged = activities_df.merge(sleep, on="calendarDate", how="inner")

# Numeric daily features
daily_cols = merged[activities_df.select_dtypes(include=[np.number]).columns].columns

# Numeric sleep features
sleep_cols = merged[add_sleep_features(sleep_df).select_dtypes(include=[np.number]).columns].columns

results = []

for s_col in sleep_cols:
    for d_col in daily_cols:

        x = merged[s_col].values.astype(float)
        y = merged[d_col].values.astype(float)

        # Remove NaN/inf pairs
        mask = np.isfinite(x) & np.isfinite(y)
        x_clean = x[mask]
        y_clean = y[mask]

        # Need at least 3 points
        if len(x_clean) < 3:
            continue

        # Skip constant arrays
        if np.nanstd(x_clean) == 0 or np.nanstd(y_clean) == 0:
            continue

        # Pearson correlation
        r, p = pearsonr(x_clean, y_clean)

        if p < 0.05:
            results.append((s_col, d_col, r, p))

# Print results
if not results:
    print("No significant sleep → daily correlations found.")
else:
    print("Significant Sleep → Daily Correlations (p < 0.05):\n")
    for s_col, d_col, r, p in sorted(results, key=lambda x: abs(x[2]), reverse=True):
        print(f"{s_col}  →  {d_col}   r={r:.3f}, p={p:.4f}")


Significant Sleep → Daily Correlations (p < 0.05):

averageSPO2  →  activity_id   r=-0.758, p=0.0000
averageSPO2  →  startTimeGmt   r=-0.758, p=0.0000
lowestSPO2  →  activity_id   r=-0.499, p=0.0000
lowestSPO2  →  startTimeGmt   r=-0.498, p=0.0000
averageRespiration  →  startTimeGmt   r=0.439, p=0.0000
averageRespiration  →  activity_id   r=0.439, p=0.0000
durationScore  →  startTimeGmt   r=0.336, p=0.0000
durationScore  →  activity_id   r=0.335, p=0.0000
decimalEndTime  →  startTimeGmt   r=0.323, p=0.0000
decimalEndTime  →  activity_id   r=0.320, p=0.0000
decimalEndTime  →  minTemperature   r=-0.276, p=0.0000
decimalEndTime  →  moderateIntensityMinutes   r=0.258, p=0.0000
centered_bed_time  →  minTemperature   r=-0.249, p=0.0000
decimalStartTime  →  minTemperature   r=-0.249, p=0.0000
lowestRespiration  →  startTimeGmt   r=0.199, p=0.0000
lowestRespiration  →  activity_id   r=0.199, p=0.0000
decimalStartTime  →  moderateIntensityMinutes   r=0.189, p=0.0000
centered_bed_time  →  modera